In [7]:
import os
import numpy as np
import pandas as pd

In [3]:
import Orange
from orangecontrib.spectroscopy import data
from orangecontrib.spectroscopy.data import getx
from Orange.data.io import FileFormat

In [4]:
def read_spectra(self):
        channels = self.get_channels()

        for c, label in channels.items():
            if label.decode("utf-8") == self.sheet:
                self.data_signal = c
                break
        else:
            self.data_signal = list(channels.keys())[0]

        import h5py
        hdf5_file = h5py.File(self.filename,'r')
        keys = list(hdf5_file.keys())

        hyperspectra = False
        intensities = []
        wavenumbers = []
        x_locs = []
        y_locs = []

        # load measurements
        for meas_name in filter(lambda s: s.startswith('Measurement'), keys):
            hdf5_meas = hdf5_file[meas_name]

            meas_keys = list(hdf5_meas.keys())
            meas_attrs = hdf5_meas.attrs

            # check if this measurement contains the selected data channel
            selected_signal = False
            for chan_name in filter(lambda s: s.startswith('Channel'), meas_keys):
                hdf5_chan = hdf5_meas[chan_name]
                if hdf5_chan.attrs.keys().__contains__('DataSignal') and hdf5_chan.attrs['DataSignal'] == self.data_signal:
                    selected_signal = True
                    break
            if not selected_signal:
                continue

            # build range arrays
            spec_vals = []
            try:
                if meas_attrs.keys().__contains__('RangeWavenumberStart'):
                    wn_start = meas_attrs['RangeWavenumberStart'][0]
                    wn_end = meas_attrs['RangeWavenumberEnd'][0]
                    wn_points = meas_attrs['RangeWavenumberPoints'][0]
                    spec_vals = np.linspace(wn_start, wn_end, wn_points)
            except:
                raise IOError("Error reading wavenumber range from " + self.filename)

            pos_vals = []
            try:
                if meas_attrs.keys().__contains__('RangeXStart'):
                    x_start = meas_attrs['RangeXStart'][0]
                    x_points = meas_attrs['RangeXPoints'][0]
                    x_incr = meas_attrs['RangeXIncrement'][0]
                    x_end = x_start + x_incr * (x_points - 1)
                    x_min = min(x_start, x_end)
                    if meas_attrs.keys().__contains__('RangeYStart'):
                        y_start = meas_attrs['RangeYStart'][0]
                        y_points = meas_attrs['RangeYPoints'][0]
                        y_incr = meas_attrs['RangeYIncrement'][0]
                        y_end = y_start + y_incr * (y_points - 1)
                        y_min = min(y_start, y_end)

                        # construct the positions array
                        for iY in range(int(y_points)):
                            y = y_min + iY * abs(y_incr)
                            for iX in range(int(x_points)):
                                x = x_min + iX * abs(x_incr)
                                pos_vals.append([x, y])
                        pos_vals = np.array(pos_vals)
                else:
                    pos_vals = np.array([1])
            except:
                raise IOError("Error reading position data from " + self.filename)

            hyperspectra = pos_vals.shape[0] > 1

            # ignore backgrounds and unchecked data
            if not hyperspectra:
                if meas_attrs.keys().__contains__('IsBackground') and meas_attrs['IsBackground'][0]:
                    continue
                if meas_attrs.keys().__contains__('Checked') and not meas_attrs['Checked'][0]:
                    continue

            if len(wavenumbers) == 0:
                wavenumbers = spec_vals

            if hyperspectra:
                x_len = meas_attrs['RangeXPoints'][0]
                y_len = meas_attrs['RangeYPoints'][0]
                x_locs = pos_vals[:x_len,0]
                y_indices = np.round(np.linspace(0, pos_vals.shape[0] - 1, y_len)).astype(int)
                y_locs = pos_vals[y_indices,1]
            else:
                x_locs.append(meas_attrs['LocationX'][0])
                y_locs.append(meas_attrs['LocationY'][0])

            # load channels
            for chan_name in filter(lambda s: s.startswith('Channel'), meas_keys):
                hdf5_chan = hdf5_meas[chan_name]
                chan_attrs = hdf5_chan.attrs

                signal = chan_attrs['DataSignal']
                if signal != self.data_signal:
                    continue

                data = hdf5_chan['Raw_Data']
                if hyperspectra:
                    rows = meas_attrs['RangeYPoints'][0]
                    cols = meas_attrs['RangeXPoints'][0]
                    intensities = np.reshape(data, (rows,cols,data.shape[1])) # organized rows, columns, wavelengths
                    break
                else:
                    intensities.append(data[0,:])

        #intensities = np.array(intensities)
        #features = np.array(wavenumbers)
        intensities = intensities
        features = np.array(wavenumbers)
        x_locs = np.array(x_locs).flatten()
        y_locs = np.array(y_locs).flatten()
        if hyperspectra:
            return _spectra_from_image(intensities, features, x_locs, y_locs)
        else:
            spectra = intensities

            # locations
            x_loc = y_loc = np.arange(len(spectra))
            metas = np.array([x_locs[x_loc], y_locs[y_loc]]).T

            domain = Orange.data.Domain([], None,
                                        metas=[Orange.data.ContinuousVariable.make("map_x"),
                                               Orange.data.ContinuousVariable.make("map_y")]
                                        )
            data = Orange.data.Table.from_numpy(domain, X=np.zeros((len(spectra), 0)),
                                                metas=np.asarray(metas, dtype=object))
            return features, spectra, data
        
Orange.data.io.FileFormat.read_spectra = read_spectra


In [11]:
files = os.listdir()
files

['MEC_02_066_FHS_AlSI_1_051023_06072023_Run2_mosaic.ptir',
 'MEC_02_068_FHS_AlSI_1_051023_06072023_Run1.ptir',
 '.ipynb_checkpoints',
 'MEC_02_066_FHS_AlSI_1_051023_06072023_Run2.ptir',
 'MEC_02_066_FHS_AlSI_1_051023_06072023_Run3.ptir',
 'Pilot_Phase_Field_Samples.ipynb']

In [24]:
spectra_list = []
for i in [1, 3, 4]:
    absolute_filename = FileFormat.locate(files[i])
    reader = FileFormat.get_reader(absolute_filename)
    spectra = read_spectra(reader)
    
    spectra_data = pd.DataFrame(spectra[1])
    spectra_data = spectra_data.drop(1).reset_index(drop=True)
    spectra_data.columns = spectra[0]
    #spectra_data_bg = spectra_data.loc[:, ~np.isnan(spectra_data.iloc[0,:])]
    spectra_data['file_name'] = files[i]
    spectra_list.append(spectra_data)
    

In [26]:
pd.concat(spectra_list)

,948.0,950.0,952.0,954.0,956.0,958.0,960.0,962.0,964.0,966.0,...,2988.0,2990.0,2992.0,2994.0,2996.0,2998.0,3000.0,3002.0,3004.0,file_name
0,32.174774,33.657639,34.959293,34.702873,34.438290,34.894096,35.138229,35.576523,35.969273,36.635277,...,4.384691,4.068508,4.315206,4.005690,3.694784,2.874610,2.473018,2.702052,3.192127,MEC_02_068_FHS_AlSI_1_051023_06072023_Run1.ptir
1,-0.611361,-0.354260,-1.053598,-1.205353,-0.474739,0.153290,0.125643,0.098530,-0.367372,-0.486663,...,-0.296220,-0.237535,-0.135871,-0.534582,-0.291247,-0.884108,0.806573,0.915925,-4.416163,MEC_02_068_FHS_AlSI_1_051023_06072023_Run1.ptir
2,-0.071959,-0.578756,0.086339,-0.898755,-0.731575,-0.595724,-0.512138,-0.328722,-0.828143,-0.230567,...,0.258327,-1.157299,-0.512236,-1.648767,-0.420439,1.172380,-0.338406,1.476841,-3.536495,MEC_02_068_FHS_AlSI_1_051023_06072023_Run1.ptir
3,2.757320,0.686351,-0.222020,-0.423446,0.433894,0.721077,0.527851,0.127807,0.066706,0.353448,...,0.326032,-0.219007,-0.793982,0.026299,0.187432,0.197198,0.727560,3.886549,7.503483,MEC_02_068_FHS_AlSI_1_051023_06072023_Run1.ptir
4,-0.154978,-0.498371,-0.010452,-0.202715,-0.343939,-1.208500,-0.811495,-0.651172,-0.252963,-0.172018,...,-0.608987,-0.640704,0.144429,0.757155,0.487847,-1.760095,-2.134103,-3.985697,-2.414529,MEC_02_068_FHS_AlSI_1_051023_06072023_Run1.ptir
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,1.092232,0.822329,-0.120623,0.330506,0.406996,0.309894,0.120296,-0.064516,-0.219138,-0.141983,...,-0.205382,-0.101822,0.313209,-0.514167,1.557830,3.302625,2.078714,0.770640,NaN,MEC_02_066_FHS_AlSI_1_051023_06072023_Run3.ptir
138,0.209352,1.434947,0.323320,0.017979,0.011899,0.556838,0.341220,0.505904,0.503907,0.376498,...,1.101542,0.638850,0.854733,0.520643,1.530720,1.072579,0.604489,0.135122,NaN,MEC_02_066_FHS_AlSI_1_051023_06072023_Run3.ptir
139,0.493740,-0.235968,0.244129,0.699546,0.244362,-0.288940,-0.222091,0.281347,0.075716,-0.194986,...,0.230600,-0.162019,0.215595,0.959821,-0.384023,-1.460202,0.181910,2.867318,NaN,MEC_02_066_FHS_AlSI_1_051023_06072023_Run3.ptir
140,11.163499,11.641426,12.196567,12.256077,12.831714,13.280186,13.542649,13.585117,13.997689,14.419927,...,0.946310,0.745126,0.175288,0.263840,0.543234,0.705321,1.565115,1.420647,NaN,MEC_02_066_FHS_AlSI_1_051023_06072023_Run3.ptir


In [32]:
spectra_list[2].columns

Index([      948.0,       950.0,       952.0,       954.0,       956.0,
             958.0,       960.0,       962.0,       964.0,       966.0,
       ...
            2986.0,      2988.0,      2990.0,      2992.0,      2994.0,
            2996.0,      2998.0,      3000.0,      3002.0, 'file_name'],
      dtype='object', length=1029)

In [35]:
spectra_list[1].columns

Index([      948.0,       950.0,       952.0,       954.0,       956.0,
             958.0,       960.0,       962.0,       964.0,       966.0,
       ...
            2988.0,      2990.0,      2992.0,      2994.0,      2996.0,
            2998.0,      3000.0,      3002.0,      3004.0, 'file_name'],
      dtype='object', length=1030)